# Approach A — Notebook 3: Assemble & Official Validation
**Amazon ML Challenge 2026: Business Entity Resolution**

### Purpose:
1. Gathers the output pickle files from Notebook 2A (`part_02a_results.pkl`) and Notebook 2B (`part_02b_results.pkl`).
2. Orders rows to precisely match official `test_source1.tsv`.
3. Exports official submission deliverables:
   - `matching_results.tsv` (1,732,544 rows)
   - `candidate_pairs.tsv` (1,732,544 rows)
4. Executes comprehensive verification checks.


In [ ]:
# 1. Environment & Path Setup
import os, sys, glob, pickle, warnings
import pandas as pd

def find_file(filename, search_root="/kaggle/input"):
    hits = glob.glob(os.path.join(search_root, "**", filename), recursive=True)
    return hits[0] if hits else None

if os.path.isdir("/kaggle/input"):
    TEST_DIR = os.path.dirname(find_file("test_source1.tsv") or "/kaggle/input/dataset/test/test_source1.tsv")
    WORK_DIR = "/kaggle/working"
else:
    TEST_DIR = os.path.abspath("data/student_resource/dataset/test")
    WORK_DIR = os.path.abspath("output/approach_a")

FINAL_DIR = os.path.join(WORK_DIR, "final_submission")
os.makedirs(FINAL_DIR, exist_ok=True)


In [ ]:
# 2. Gather Partition 02a & 02b Results
result_files = sorted(glob.glob("/kaggle/input/**/part_02*_results.pkl", recursive=True) or
                      glob.glob(os.path.join(WORK_DIR, "**", "part_02*_results.pkl"), recursive=True) or
                      glob.glob("/kaggle/input/**/shard_*_results.pkl", recursive=True) or
                      glob.glob(os.path.join(WORK_DIR, "**", "shard_*_results.pkl"), recursive=True))

print(f"Discovered {len(result_files)} result files: {result_files}")
assert len(result_files) > 0, "No partition files found! Attach datasets containing part_02a_results.pkl and part_02b_results.pkl"

all_matches = {}
all_cands = {}

for rf in result_files:
    print(f"Loading {rf}...")
    with open(rf, "rb") as f:
        data = pickle.load(f)
    all_matches.update(data["matches"])
    all_cands.update(data["cands"])

print(f"Aggregated predictions for {len(all_matches):,} unique S1 entities")


In [ ]:
# 3. Export Deliverables to Official Competition Format
s1_test_path = os.path.join(TEST_DIR, "test_source1.tsv")
s1_ids_order = pd.read_csv(s1_test_path, sep="	", dtype=str, usecols=["entity_id"])["entity_id"].tolist()
total_expected = len(s1_ids_order)
print(f"Official test S1 expects exactly {total_expected:,} entities")

assert len(all_matches) == total_expected, f"Entity count mismatch! Aggregated {len(all_matches):,}, expected {total_expected:,}"

mp = os.path.join(FINAL_DIR, "matching_results.tsv")
cp = os.path.join(FINAL_DIR, "candidate_pairs.tsv")

print(f"Writing {mp}...")
with open(mp, "w", encoding="utf-8") as f:
    f.write("source1_entity_id	matched_entity_ids
")
    for sid in s1_ids_order:
        f.write(f"{sid}	{','.join(all_matches.get(sid, []))}
")

print(f"Writing {cp}...")
with open(cp, "w", encoding="utf-8") as f:
    f.write("source1_entity_id	candidate_entity_ids
")
    for sid in s1_ids_order:
        f.write(f"{sid}	{','.join(all_cands.get(sid, []))}
")

n_matched = sum(1 for sid in s1_ids_order if all_matches.get(sid, []))
n_preds = sum(len(all_matches.get(sid, [])) for sid in s1_ids_order)
print(f"Summary: {n_matched:,}/{total_expected:,} entities predicted with matches ({n_preds:,} total pairs)")


In [ ]:
# 4. Comprehensive Validation Harness
s2_ids = set(pd.read_csv(os.path.join(TEST_DIR, "test_source2.tsv"), sep="	", dtype=str, usecols=["entity_id"])["entity_id"])
s3_ids = set(pd.read_csv(os.path.join(TEST_DIR, "test_source3.tsv"), sep="	", dtype=str, usecols=["entity_id"])["entity_id"])
valid_targets = s2_ids | s3_ids

mdf = pd.read_csv(mp, sep="	", dtype=str).fillna("")
errors = []

if len(mdf) != total_expected:
    errors.append(f"Row count mismatch: got {len(mdf):,}, expected {total_expected:,}")

if mdf["source1_entity_id"].duplicated().any():
    errors.append("Duplicate source1_entity_id entries detected in matching_results.tsv!")

for idx, row in mdf.iterrows():
    mids = [x for x in row["matched_entity_ids"].split(",") if x]
    if len(mids) != len(set(mids)):
        errors.append(f"Duplicate IDs inside matched list for {row['source1_entity_id']}")
        break
    bad = [x for x in mids if x not in valid_targets]
    if bad:
        errors.append(f"Invalid target IDs for {row['source1_entity_id']}: {bad[:3]}")
        break

if not errors:
    print("="*60)
    print("ALL VALIDATION CHECKS PASSED!")
    print(f"Primary output ready: {mp}")
    print(f"Candidate output ready: {cp}")
    print("="*60)
else:
    print(f"VALIDATION FAILED with errors: {errors}")
